# 01. データ準備

BC5CDrデータセットをダウンロードして、NERのためのデータ準備を行います。

In [1]:
import torch
import spacy
import string
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA利用可: {torch.cuda.is_available()}")

PyTorch: 2.5.1+cu121
CUDA利用可: True


In [2]:
# spaCy トークナイザー
nlp = spacy.blank("en")
print("spaCy準備完了")

spaCy準備完了


In [3]:
# bigbio/bc5cdr をロード
dataset = load_dataset("bigbio/bc5cdr", name="bc5cdr_bigbio_kb", trust_remote_code=True)
print("データセットロード完了")
print(dataset)

データセットロード完了
DatasetDict({
    train: Dataset({
        features: ['id', 'document_id', 'passages', 'entities', 'events', 'coreferences', 'relations'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'document_id', 'passages', 'entities', 'events', 'coreferences', 'relations'],
        num_rows: 500
    })
    validation: Dataset({
        features: ['id', 'document_id', 'passages', 'entities', 'events', 'coreferences', 'relations'],
        num_rows: 500
    })
})


In [4]:
# ラベル定義
label_list = ['O', 'B-Chemical', 'I-Chemical', 'B-Disease', 'I-Disease']
label2id = {label: i for i, label in enumerate(label_list)}

print("=== ラベル定義 ===")
for i, label in enumerate(label_list):
    print(f"{i}: {label}")

=== ラベル定義 ===
0: O
1: B-Chemical
2: I-Chemical
3: B-Disease
4: I-Disease


In [5]:
# bigbio → IOB2 変換
ner_data = {'train': [], 'validation': [], 'test': []}

for split in ['train', 'validation', 'test']:
    for sample in dataset[split]:
        entities = sample['entities']
        
        for passage in sample['passages']:
            text = passage['text'][0]
            passage_offset = passage['offsets'][0][0]
            
            doc = nlp(text)
            labels = [0] * len(doc)
            
            for ent in entities:
                start, end = ent['offsets'][0]
                rel_start = start - passage_offset
                rel_end = end - passage_offset
                
                if rel_start >= 0:
                    for i, token in enumerate(doc):
                        if token.idx >= rel_start and token.idx + len(token.text) <= rel_end:
                            prefix = "B" if token.idx == rel_start else "I"
                            labels[i] = label2id[f"{prefix}-{ent['type']}"]
            
            # 対処: 単一文字の記号を除外
            for i, token in enumerate(doc):
                if len(token.text) == 1 and token.text in string.punctuation + string.whitespace:
                    labels[i] = 0
            
            tokens = [t.text for t in doc]
            ner_data[split].append({'tokens': tokens, 'tags': labels})

print("=== 変換完了 ===")
for split in ner_data:
    print(f"{split}: {len(ner_data[split])} サンプル")

=== 変換完了 ===
train: 1000 サンプル
validation: 1000 サンプル
test: 1000 サンプル


In [6]:
# 結果確認（Chemical + Disease）
chemical_found = False
disease_found = False

for i in range(min(20, len(ner_data['train']))):
    sample = ner_data['train'][i]
    tokens = sample['tokens']
    tags = sample['tags']
    
    has_chemical = any(tag in [1, 2] for tag in tags)
    has_disease = any(tag in [3, 4] for tag in tags)
    
    if has_chemical and not chemical_found:
        chemical_found = True
        print(f"=== Sample {i}: Chemical ===")
        for t, tag in zip(tokens, tags):
            if tag != 0:
                print(f"  {t} [{label_list[tag]}] 🔵")
    
    if has_disease and not disease_found:
        disease_found = True
        print(f"\n=== Sample {i}: Disease ===")
        for t, tag in zip(tokens, tags):
            if tag != 0:
                print(f"  {t} [{label_list[tag]}] 🟠")
    
    if chemical_found and disease_found:
        break

=== Sample 0: Chemical ===
  Naloxone [B-Chemical] 🔵
  clonidine [B-Chemical] 🔵

=== Sample 1: Disease ===
  hypertensive [B-Disease] 🟠
  clonidine [B-Chemical] 🟠
  nalozone [B-Chemical] 🟠
  hypotensive [B-Disease] 🟠
  alpha [B-Chemical] 🟠
  methyldopa [I-Chemical] 🟠
  naloxone [B-Chemical] 🟠
  Naloxone [B-Chemical] 🟠
  hypertensive [B-Disease] 🟠
  clonidine [B-Chemical] 🟠
  3H]-naloxone [I-Chemical] 🟠
  naloxone [B-Chemical] 🟠
  clonidine [B-Chemical] 🟠
  3H]-dihydroergocryptine [I-Chemical] 🟠
  hypertensive [B-Disease] 🟠
  naloxone [B-Chemical] 🟠
  clonidine [B-Chemical] 🟠
  clonidine [B-Chemical] 🟠
  alpha [B-Chemical] 🟠
  methyldopa [I-Chemical] 🟠


In [7]:
# BioBERT トークナイザー
model_name = "dmis-lab/biobert-v1.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"トークナイザー: {model_name}")

トークナイザー: dmis-lab/biobert-v1.1


In [8]:
# 前処理
def tokenize_fn(examples):
    tokenized = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, tag in enumerate(examples["tags"]):
        word_ids = tokenized.word_ids(i)
        label_ids = []
        prev = None
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)
            elif wid != prev:
                label_ids.append(tag[wid])
            else:
                label_ids.append(-100)
            prev = wid
        labels.append(label_ids)
    tokenized["labels"] = labels
    return tokenized

tokenized_datasets = {}
for split in ['train', 'validation', 'test']:
    ds = Dataset.from_list(ner_data[split])
    tokenized_datasets[split] = ds.map(tokenize_fn, batched=True, remove_columns=['tokens', 'tags'])

final_datasets = DatasetDict({
    'train': tokenized_datasets['train'],
    'validation': tokenized_datasets['validation'],
    'test': tokenized_datasets['test']
})

print("前処理完了")

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

前処理完了


In [9]:
# 保存
final_datasets.save_to_disk("processed_bc5cdr")

print("=== 保存完了 ===")
print("保存先: notebooks/processed_bc5cdr")
print(f"ラベル: {label_list}")
print("\n次: 02_model_training.ipynb")

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

=== 保存完了 ===
保存先: notebooks/processed_bc5cdr
ラベル: ['O', 'B-Chemical', 'I-Chemical', 'B-Disease', 'I-Disease']

次: 02_model_training.ipynb
